# Task 02: Generate Classification Dataset with LLM

Use `gpt-4o-mini` to annotate cybersecurity texts for GLiClass training.

Attack types: `ransomware`, `phishing`, `apt_attack`, `ddos`, `data_breach`, `supply_chain_attack`, `zero_day`

You will:
1. Call LLM to classify a text and get both true and false labels as JSON
2. Implement `build_gliclass_example(client, text)` → `{text, all_labels, true_labels}`
3. Generate and validate the full dataset from all 20 reports

In [ ]:
import json, os
from openai import OpenAI
from collections import Counter

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "raw_security_reports.json")) as f:
    reports = json.load(f)

api_key = "your-api-key-here"  # Replace with your actual key
client = OpenAI(api_key=api_key)

ALL_ATTACK_LABELS = [
    "ransomware", "phishing", "apt_attack", "ddos",
    "data_breach", "supply_chain_attack", "zero_day"
]
print(f"✓ Loaded {len(reports)} reports")

## Part 1: Call LLM and Parse JSON Response

Write a system prompt that returns JSON:
```json
{"true_labels": ["ransomware"], "false_labels": ["phishing", "ddos"]}
```

Implement `call_llm_cls(client, text)` using `response_format={"type": "json_object"}`.
Store first report result in `cls_result`.

In [ ]:
# YOUR CODE HERE
test_text = reports[0]['text']

# SYSTEM_PROMPT = ...

# def call_llm_cls(client, text):
#     ...

# cls_result = ...


In [ ]:
# TEST — Part 1
assert 'cls_result' in vars(), "Store result in 'cls_result'"
assert isinstance(cls_result, dict), "cls_result must be a dict (parsed JSON)"
assert 'true_labels' in cls_result, "Response must contain 'true_labels'"
assert 'false_labels' in cls_result, "Response must contain 'false_labels'"
assert len(cls_result['true_labels']) >= 1, "Need at least 1 true label"
assert len(cls_result['false_labels']) >= 1, "Need at least 1 false label"
# Check labels are valid
for label in cls_result['true_labels'] + cls_result['false_labels']:
    assert label in ALL_ATTACK_LABELS, f"Unknown label: {label!r}"
# Report 0 (LockBit ransomware) must include 'ransomware'
assert 'ransomware' in cls_result['true_labels'], \
    f"Report 0 (LockBit ransomware) must have 'ransomware' in true_labels, got {cls_result['true_labels']}"
print(f"✓ true_labels:  {cls_result['true_labels']}")
print(f"  false_labels: {cls_result['false_labels']}")

## Part 2: Implement `build_gliclass_example`

Implement `build_gliclass_example(client, text)` that returns:
```python
{"text": str, "all_labels": list, "true_labels": list}
```
Where `all_labels = true_labels + false_labels` (deduplicated).
Return `None` if no true labels found or labels are invalid.

In [ ]:
# YOUR CODE HERE
def build_gliclass_example(client, text):
    """
    Annotate text for GLiClass training.
    Returns {text, all_labels, true_labels} or None.
    """
    pass


In [ ]:
# TEST — Part 2
ex = build_gliclass_example(client, reports[1]['text'])  # phishing report
assert ex is not None
assert {'text', 'all_labels', 'true_labels'} <= ex.keys()
assert ex['text'] == reports[1]['text']
assert len(ex['true_labels']) >= 1
assert set(ex['true_labels']).issubset(set(ex['all_labels'])), \
    "true_labels must be a subset of all_labels"
for label in ex['all_labels']:
    assert label in ALL_ATTACK_LABELS, f"Invalid label: {label}"
assert 'phishing' in ex['true_labels'], \
    f"Report 1 (phishing) must have 'phishing' in true_labels, got {ex['true_labels']}"
print(f"✓ build_gliclass_example works")
print(f"  all_labels:  {ex['all_labels']}")
print(f"  true_labels: {ex['true_labels']}")

## Part 3: Generate Full Dataset

Annotate all 20 reports. Store in `cls_dataset`. Validate and save to `fixtures/expected/cls_dataset.json`.

Requirements:
- At least 15 valid examples
- `true_labels ⊆ all_labels` for every example
- At least 4 different attack types represented

In [ ]:
# YOUR CODE HERE
from tqdm import tqdm

# def build_cls_dataset(client, reports):
#     ...

# cls_dataset = ...


In [ ]:
# TEST — Part 3
assert 'cls_dataset' in vars()
assert len(cls_dataset) >= 15, f"Expected ≥15 examples, got {len(cls_dataset)}"

all_true_labels = set()
for ex in cls_dataset:
    assert {'text', 'all_labels', 'true_labels'} <= ex.keys()
    assert ex['true_labels'], "empty true_labels"
    assert set(ex['true_labels']).issubset(set(ex['all_labels']))
    for label in ex['all_labels']:
        assert label in ALL_ATTACK_LABELS
    all_true_labels.update(ex['true_labels'])

assert len(all_true_labels) >= 4, f"Need ≥4 attack types, got: {all_true_labels}"

save_path = os.path.normpath(os.path.join(fixtures, "..", "expected", "cls_dataset.json"))
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, 'w') as f:
    json.dump(cls_dataset, f, indent=2)

label_counts = Counter(l for ex in cls_dataset for l in ex['true_labels'])
print(f"✓ Generated {len(cls_dataset)} classification examples")
print(f"  Attack types found: {sorted(all_true_labels)}")
print(f"  Label distribution: {dict(label_counts.most_common())}")
print(f"  Saved to fixtures/expected/cls_dataset.json")